# Compare transfer learning training regimes

Build five reproducible checks for a small classification training loop. The bonus Colab lab learns a shared feature representation on a source task and compares training a new head with fine-tuning the same representation on a shifted target.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Check a classification batch

Check feature and label shapes.

Implement valid_batch(features, labels, width, n_classes). Require a nonempty batch, rectangular rows of the given width and one integer class ID per row in [0, n_classes). Reject boolean labels.


In [ ]:
def valid_batch(features, labels, width, n_classes):
    raise NotImplementedError


In [ ]:
assert valid_batch([[1,2],[3,4]],[0,1],2,2)
assert not valid_batch([[1,2],[3]],[0,1],2,2)
assert not valid_batch([[1,2]],[True],2,2)
assert not valid_batch([[1,2]],[2],2,2)
assert not valid_batch([],[],2,2)
assert not valid_batch([[1,2],[3,4]],[0],2,2), "Reject a batch with fewer labels than rows"
assert not valid_batch([[1,2]],[-1],2,2), "Reject negative class IDs: valid IDs are 0 to n_classes - 1"
print("OK")


## 2. Aggregate uneven validation batches

Aggregate validation loss correctly.

Implement mean_validation_loss(batches). Each tuple contains a nonnegative batch size and a finite per-example mean loss. Return the global mean, or None when there are no contributing examples.


In [ ]:
def mean_validation_loss(batches):
    raise NotImplementedError


In [ ]:
assert abs(mean_validation_loss([(4,0.2),(1,1.0)]) - 0.36) < 1e-12
assert mean_validation_loss([(0,9),(2,0.5)]) == 0.5
assert mean_validation_loss([]) is None
assert mean_validation_loss([(0,1)]) is None
print("OK")


## 3. Select a validation checkpoint

Choose the best checkpoint using validation only.

Implement best_epoch(history) for (epoch, validation_loss) tuples. Return the lowest-loss epoch and choose the smallest epoch on ties. Return None for an empty history.


In [ ]:
def best_epoch(history):
    raise NotImplementedError


In [ ]:
assert best_epoch([(4,0.3),(2,0.3),(1,0.8)]) == 2
assert best_epoch([(1,0.5),(2,0.6)]) == 1
assert best_epoch([(9,0.1)]) == 9
assert best_epoch([]) is None
print("OK")


## 4. Build a parameter-group plan

Compare frozen-head and fine-tuning parameter groups.

Implement learning_rate_plan(groups, head_rate, backbone_rate). Groups maps names to head or backbone. Include only names with positive group rates. Reject unknown group labels, even if other rates are zero.


In [ ]:
def learning_rate_plan(groups, head_rate, backbone_rate):
    raise NotImplementedError


In [ ]:
assert learning_rate_plan({"a":"head","b":"backbone"},0.1,0) == {"a":0.1}
assert learning_rate_plan({"a":"head","b":"backbone"},0.1,0.01) == {"a":0.1,"b":0.01}
assert learning_rate_plan({},0.1,0) == {}
try:
    learning_rate_plan({"a":"other"},0,0)
except ValueError:
    pass
else:
    raise AssertionError("Reject unknown groups")
print("OK")


## 5. Select a usable checkpoint from batch summaries

Implement checkpoint_report(epochs). Each row is (epoch, validation batch summaries). Exclude
epochs without any contributing examples. Return (ordered usable epoch/mean pairs in input order,
best epoch or None). Use mean_validation_loss and best_epoch. Test that unequal batches can change
which epoch wins.


In [ ]:
def checkpoint_report(epochs):
    raise NotImplementedError


In [ ]:
history, best = checkpoint_report([(1,[(4,0.2),(1,1.0)]),(2,[(5,0.4)]),(3,[])])
assert len(history) == 2 and abs(history[0][1] - 0.36) < 1e-12 and best == 1
assert checkpoint_report([(8,[(2,0.5)]),(4,[(1,0.5)])])[1] == 4
assert checkpoint_report([(1,[(0,4)])]) == ([],None)
print("OK")


## Interpretation

Describe an experiment that isolates whether transfer helps. Specify a common target validation split, a scratch baseline, and the evidence that would justify unfreezing the backbone.

**Your answer**


## Bonus lab: compare a frozen head and fine-tuning in PyTorch

Run the notebook in Colab with PyTorch installed. CPU is enough and no dataset download is needed.
First fit a source classifier with a learned backbone. Then compare scratch training, a frozen
pretrained backbone with a new head, and a run that unfreezes the same backbone at step 41.
The target uses an independently sampled dataset and changes the classification boundary.

All three target runs share the same data, initialisation for the new head, 80 optimiser steps
and checkpoint selection rule. The fine-tune run rebuilds its optimiser when it unfreezes, so
optimiser reset is part of that configuration. Source training is additional compute. Evaluate
only the selected checkpoint on test. Report changes in validation curves, training cost and
what a repeat across seeds would establish. No particular candidate is required to win.


In [ ]:
# Colab already includes torch; locally: !pip install torch
import importlib.util

if importlib.util.find_spec("torch") is None:
    print("Optional lab skipped. Open this notebook in Colab with PyTorch.")
else:
    from copy import deepcopy

    import torch
    from torch import nn
    from torch.nn import functional as F

    def fixture(size, seed, shift):
        generator = torch.Generator().manual_seed(seed)
        latent = torch.randn(size, 2, generator=generator)
        noisy = 0.15 * torch.randn(size, 4, generator=generator)
        features = torch.cat((latent, latent), dim=1) + noisy
        target = (latent[:, 0] + shift * latent[:, 1] > 0).long()
        return features, target

    source = fixture(512, seed=1, shift=0.0)
    train = fixture(64, seed=2, shift=0.8)
    validation = fixture(128, seed=3, shift=0.8)
    test = fixture(256, seed=4, shift=0.8)

    def backbone():
        return nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())

    def train_epoch(model, batch, optimizer, frozen):
        model.train()
        if frozen:
            model[0].eval()
        optimizer.zero_grad()
        loss = F.cross_entropy(model(batch[0]), batch[1])
        loss.backward()
        optimizer.step()

    @torch.no_grad()
    def loss_and_accuracy(model, batch):
        model.eval()
        outputs = model(batch[0])
        return F.cross_entropy(outputs, batch[1]).item(), (outputs.argmax(1) == batch[1]).float().mean().item()

    torch.manual_seed(17)
    source_model = nn.Sequential(backbone(), nn.Linear(4, 2))
    source_optimizer = torch.optim.Adam(source_model.parameters(), lr=0.02)
    for _ in range(80):
        train_epoch(source_model, source, source_optimizer, frozen=False)

    torch.manual_seed(23)
    head_template = nn.Linear(4, 2)
    scratch = nn.Sequential(backbone(), deepcopy(head_template))
    frozen_model = nn.Sequential(deepcopy(source_model[0]), deepcopy(head_template))
    fine_tuned = deepcopy(frozen_model)

    def optimizer_for(model, frozen, backbone_lr=0.003):
        model[0].requires_grad_(not frozen)
        groups = [{"params": model[1].parameters(), "lr": 0.03}]
        if not frozen:
            groups.append({"params": model[0].parameters(), "lr": backbone_lr})
        return torch.optim.Adam(groups)

    records = {}
    for name, model in [("scratch", scratch), ("frozen", frozen_model), ("fine_tune", fine_tuned)]:
        frozen = name != "scratch"
        initial_backbone = deepcopy(model[0].state_dict())
        # Scratch trains its whole network at the head rate; fine-tuning uses a smaller backbone rate.
        optimizer = optimizer_for(model, frozen, backbone_lr=0.03 if name == "scratch" else 0.003)
        checkpoints = {}
        history = []
        for step in range(1, 81):
            if name == "fine_tune" and step == 41:
                frozen = False
                optimizer = optimizer_for(model, frozen)
            train_epoch(model, train, optimizer, frozen)
            if step % 10 == 0:
                val_loss, _ = loss_and_accuracy(model, validation)
                history.append((step, val_loss))
                checkpoints[step] = deepcopy(model.state_dict())
        if name == "frozen":
            assert all(torch.equal(value, model[0].state_dict()[key]) for key, value in initial_backbone.items())
        picked = best_epoch(history)
        model.load_state_dict(checkpoints[picked])
        records[name] = (min(value for _, value in history), picked, model)

    chosen = min(records, key=lambda name: (records[name][0], name))
    for name, (val_loss, step, _) in records.items():
        print(f"{name:10s} validation loss {val_loss:.3f} at step {step}")
    test_loss, test_accuracy = loss_and_accuracy(records[chosen][2], test)
    print(f"Selected by validation: {chosen}; test loss {test_loss:.3f}, test accuracy {test_accuracy:.3f}")
